In [1]:
%matplotlib inline
import seaborn as sns
import os
import sys
import subprocess
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from pyarrow import fs
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

os.environ["HADOOP_CONF_DIR"] = "/path/to/hadoop/conf/"
os.environ["JAVA_HOME"] = "/path/to/java"
os.environ["HADOOP_HOME"] = "/path/to/hadoop"
os.environ["ARROW_LIBHDFS_DIR"] = "/path/to/hadoop/lib/"
os.environ["CLASSPATH"] = subprocess.check_output(
    "$HADOOP_HOME/bin/hadoop classpath --glob", shell=True
).decode("utf-8")

hdfs = fs.HadoopFileSystem(
    host="hdfs://your-hdfs-host", port=8020
)

# import pandas as pd
from tqdm import tqdm
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)
import sys

if not sys.warnoptions:
    import warnings

    warnings.simplefilter("ignore")

import logging

logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").disabled = True

df_total = pd.read_parquet(
    "/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/total_sl_13_provinces.parquet",
    filesystem=hdfs, )
df_total.sort_index(ascending=False).head()
# df_total = df_total[df_total.index<'2023-7-1']


2024-07-18 10:22:31,651 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


FileNotFoundError: /path/to/company_data/sa/cpc/store/data/SUM_TOTAL/total_sl_13_provinces.parquet

In [2]:
df_sl_13province = pd.DataFrame()
for year in range(2014,2025):
    df_each_year = pd.read_parquet("/path/to/company_data/user/<user>/CPC/SL_MDD_1M.parquet/Y={}".format(year),
                                   filesystem=hdfs)
    df_each_year['year'] = year
    print(df_each_year.shape)
    df_sl_13province = pd.concat([df_sl_13province,df_each_year],axis=0)
    # break
df_sl_13province['date'] = df_sl_13province.apply(lambda row: f"{row['year']}-{row['M']}-1",axis=1)
df_sl_13province['date'] = pd.to_datetime(df_sl_13province['date'])
df_sl_13province = df_sl_13province.pivot_table(index='date',columns='MA_DDO',values='SAN_LUONG')
df_total = df_sl_13province

(9307, 13)
(10498, 13)
(11900, 13)
(13273, 13)
(14744, 13)
(16408, 13)
(17825, 13)
(18868, 13)
(19212, 13)
(20076, 13)
(1666, 13)


In [3]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from prophet import Prophet
import pandas as pd
from pmdarima import auto_arima


def measure_simility(arr1, arr2):
    arr1 = arr1.reshape((-1,))
    arr2 = arr2.reshape((-1,))
    score = np.corrcoef(arr1, arr2)[0, 1]
    return score


def pick_data_train(vt, arrs, n):
    score = []
    for arr in arrs:
        score.append(measure_simility(vt, arr))
    score = np.array(score)
    return np.argsort(score)[-n:], score[np.argsort(score)[-n:]]


def create_data_total(df, month, year, lag, shift):
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].values.reshape((-1, lag))
    # y_known = df[df.index == year].iloc[0,11 + month-lag: 11 +month].values
    y_known = df[df.index == year].iloc[0, 11 + month - lag: 11 + month].values
    # y_known = df.iloc[-1, 11 + month - lag: 11 + month].values
    x_news = df[df.index < year].iloc[:, 11 + month - shift].values
    # print(month,year,y_known)
    return x_knowns, y_known, x_news


def create_data_cumsum(df, month, year, lag, shift):
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].cumsum(axis=1).values.reshape(
        (-1, lag))
    y_known = df[df.index == year].iloc[:, 11 + month - lag: 11 + month].cumsum(axis=1).iloc[0, :].values
    # y_known = df.iloc[[-1], 11 + month - lag: 11 + month].cumsum(axis=1).iloc[0, :].values
    x_news = df[df.index < year].iloc[:, 11 + month - lag - shift: 12 + month - shift].cumsum(axis=1).iloc[:, -1].values
    return x_knowns, y_known, x_news


def create_data_rolling(df, month, year, lag, shift, n_rolling):
    y_old = df[df.index == year].iloc[0, 11 + month - 1 - n_rolling + 1:11 + month - 1].sum()
    df = df.rolling(n_rolling, axis=1).mean()
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].values.reshape((-1, lag))
    y_known = df[df.index == year].iloc[0, 11 + month - lag: 11 + month].values
    x_news = df[df.index < year].iloc[:, 11 + month - shift].values
    return x_knowns, y_known, x_news, y_old


def predict_total(known_y, known_x, new_x, order):
    # known_y = known_y.reshape((-1,))
    # known_x = known_x.reshape((-1,))
    coefficients = np.polyfit(known_x, known_y, order)
    estimated_func = np.poly1d(coefficients)
    return float(estimated_func(new_x))


def predict_cumsum(known_y, known_x, new_x, order):
    # known_y = known_y.reshape((-1,))
    # known_x = known_x.reshape((-1,))
    coefficients = np.polyfit(known_x, known_y, order)
    estimated_func = np.poly1d(coefficients)
    return float(estimated_func(new_x))


def predict(df_total, year, month: int, lag: int, order1: int, order2: int, shift: int, n_max):
    x_knowns, y_known, x_news = create_data_total(df_total, month, year, lag, shift)
    max_scores, coeffs = pick_data_train(y_known, x_knowns, n_max)
    pre = []
    for max_score in max_scores:
        x_knowns, y_known, x_news = create_data_total(df_total, month, year, lag, shift)
        x_known = x_knowns[max_score]
        x_new = x_news[max_score]

        pred1 = predict_total(known_y=y_known,
                              known_x=x_known,
                              new_x=x_new, order=order1)
        pre.append(pred1)
        shift = 0
        x_knowns, y_known, x_news = create_data_cumsum(df_total, month, year, lag, shift)
        x_known = x_knowns[max_score]
        x_new = x_news[max_score]
        pred2 = predict_cumsum(known_y=y_known,
                               known_x=x_known,
                               new_x=x_new, order=order2) - y_known[-1]

        pre.append(pred2)
    return np.mean(pre), coeffs.mean()


def predict_master(data: pd.DataFrame, d_date: pd.DataFrame, month, year, lag, p, d, q, n_max, type_decomp,
                   methods_season, method_resid, method_trend,order1,order2):
    date = f"{year}-{month}-1"
    # print(date)
    # d_train = data[data.index < f"{year}-1-1"]
    d_train = data[data.index < date]
    d_train = pd.concat([d_train, d_date], axis=1).dropna(axis=0)

    d_residuals, d_trend, d_seasonal = detrend(d_train, type=type_decomp)
    # y_residual=0
    y_residual = forecast_residual(d_residuals.values, p, d, q, method_resid=method_resid)
    # except:
    #     y_residual = 0

    d_seasonal.name = name_province
    # d_seasonal = pd.concat([d_seasonal, d_date], axis=1)
    d_seasonal = pd.concat([d_seasonal, d_date], axis=1).fillna(0)
    d_seasonal = d_seasonal.pivot_table(columns='month', index='year', values=name_province)
    y_seasonal = forecast_seasonal(d_seasonal, month, year, lag, n_max,order1,order2, methods_season)

    d_trend.name = name_province
    d_trend = pd.concat([d_trend, d_date], axis=1).dropna()
    y_trend = forecast_trend(d_trend.iloc[-2, 0], d_trend.iloc[-1, 0], method_trend=method_trend)

    # d_residuals = pd.concat([d_residuals, d_date], axis=1).dropna()
    # y_re3 = d_seasonal.pivot_table(columns='month', index='year', values=name_province).dropna(axis=0).mean(axis=0)[
    #     month]
    # d_residuals = d_residuals.pivot_table(columns='month', index='year', values=name_province)
    # d_trend = d_trend.pivot_table(columns='month', index='year', values=name_province)
    # y_re2 = forecast_seasonal(d_trend, month, year, lag, 2)
    # return y_lr + (y_re1 +y_re2+y_re)/3,y_re
    # return (y_re3 + y_re1) / 2 + y_re2
    return y_trend, y_seasonal, y_residual


def detrend(df: pd.DataFrame, type='STL'):
    if type == 'STL':
        data = df.copy()
        data = data.iloc[:, 0]
        stl = STL(data,period=12)
        res = stl.fit()
        trend = res.trend
        resid = res.resid
        season = res.seasonal
        return resid, trend, season
    if type == 'prophet':
        data = df.copy()
        data = data.iloc[:, [0]]
        data = data.reset_index()
        data.columns = ['ds', 'y']
        model = Prophet(yearly_seasonality=12)
        model.fit(data)
        forecaster = model.predict(model.make_future_dataframe(periods=0, freq='MS'))
        forecaster['y'] = data['y']
        forecaster['resid'] = forecaster.apply(lambda row: (row.y - (row.yearly + row.trend)), axis=1)
        forecaster = forecaster.set_index('ds')
        df_resid = forecaster['resid']
        df_trend = forecaster['trend']
        df_season = forecaster['yearly']
        df_season.name = 'seasonal'
        return df_resid, df_trend, df_season
    else:
        return None, None, None


def compute_error(data, month, year, y_pr):
    y_tr = data[data.index == f'{year}-{month}-1'].values[0]
    return (y_pr - y_tr) / y_tr, y_tr


def forecast_seasonal(data, month, year, lag, n_max,order1,order2, method='statistics'):
    if method == 'statistics':
        df = data.copy()
        df = pd.concat([df.shift(1, axis=0), df, df.shift(-1, axis=0)], axis=1).iloc[1:, :] + 10e10
        # lag = int(params[month - 1][1])
        # shift = int(params[month - 1][2])
        # order1 = int(params[month - 1][3])
        # order2 = int(params[month - 1][4])
        # lag = 6
        shift = 0
        # order1 = 1
        # order2 = 1
        y, _ = predict(df, year, month, lag, order1, order2, shift, n_max)
        return y - 10e10
    if method == 'average':
        df = data.copy()
        # df = df[df.index<year]
        # print(df.dropna(axis=0).mean(axis=0))
        y = df.dropna(axis=0).mean(axis=0)[month]
        # print(y)
        return y
    if method == 'latest':
        df = data.copy()
        # df = df[df.index<year]
        # print(df.dropna(axis=0).mean(axis=0))
        y = df.dropna(axis=0).iloc[-1, :][month]
        # print(y)
        return y
    else:
        raise "Not specify the season method"


def forecast_trend(y_1, y_t, method_trend='average'):
    if method_trend == 'average':
        y_pr_trend = y_t + (y_t - y_1)
        # y_pr_trend = y_t
        return y_pr_trend
    if method_trend == 'latest':
        return y_t
    else:
        raise "Not specify the trend method"


def forecast_residual(series, p, d, q, method_resid='arima'):
    if method_resid == 'arima':
        try:
            model = ARIMA(series, order=(p, d, q))
            model_fit = model.fit()
            return model_fit.forecast(freq='MS')[0]
        except:return 0
    # if method_resid == 'autoarima':
    #     model = auto_arima(series, seasonal=False, trace=False)
    #     return model.predict(n_periods=1, freq='MS')[0]
    else:
        return 0


def get_ytr(df, month, year):
    try:
        filter = f'{year}-{month}-01'
        return df[df.index == filter].values[0][0]
    except:
        return None


In [41]:
df_sl_13province = pd.read_parquet(
    "/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/group1_2024-05.parquet",
    filesystem=hdfs, )

# df_total_1 = df_sl_13province.copy().iloc[:, :-2]
df_total_1 = df_sl_13province.copy()
df_total_1["PHIEN"] = pd.to_datetime(df_total_1["PHIEN"])
df_total_1 = df_total_1.set_index('PHIEN')
df_total_1.sort_index(ascending=False)

,BDINH,DANANG,DLAC,DNONG,GLAI,KHOA,KTUM,PYEN,QBINH,QNAM,QNGAI,QTRI,TTHUE
PHIEN,,,,,,,,,,,,,
2024-04-01,67136921,87510200,34560542,14894298,15262132,69391199,8464348,13308806,17232980,87506935,111399448,12632028,59565689
2024-03-01,71818467,92598565,37446673,14297373,17644288,65358222,14474257,13711591,17772348,85721820,94975556,13253699,59463922
2024-02-01,46307001,57348267,17832089,11107772,12247593,52374313,8693606,8222259,9703018,56907292,74151200,8689612,39367917
2024-01-01,71595870,79815091,31651999,13631752,17263617,62594354,17375190,12721019,13837842,80103464,76011178,12955801,51077020
2023-12-01,72731563,80123953,31352540,17193717,16066089,60591905,14312237,12058416,15138735,80961967,74675587,14829097,62842799
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-07-01,14755116,54398181,4203784,2035191,121529091,38613560,725146,2990314,2319993,15231470,10587578,5334937,17456535
2014-06-01,18076060,54372438,4519714,1931883,2441064,38306096,542470,4356837,2352153,15582064,12262141,7151433,18747506
2014-05-01,15931720,50055621,4389944,2780161,3022910,36582588,1045780,5482095,2126144,13847947,10958768,5077516,15963621


In [13]:
df_sl_13province = pd.read_parquet(
    "/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/group2_2024-05.parquet",
    filesystem=hdfs, )

# df_total_1 = df_sl_13province.copy().iloc[:, :-2]
df_total_2 = df_sl_13province.copy()
df_total_2["PHIEN"] = pd.to_datetime(df_total_2["PHIEN"])
df_total_2 = df_total_2.set_index('PHIEN')
df_total_2.sort_index(ascending=False)

,BDINH,DANANG,DLAC,DNONG,GLAI,KHOA,KTUM,PYEN,QBINH,QNAM,QNGAI,QTRI,TTHUE
PHIEN,,,,,,,,,,,,,
2024-04-01,177204123,221139552,220238931,61634126,151020897,194551034,41416013,88093957,87320736,145863869,110559294,58680042,127099508
2024-03-01,154604527,192366318,218178932,65392892,143076192,170333301,38832327,77578536,68501270,133263920,100213704,52441930,103890813
2024-02-01,146269039,150103083,179669813,59246135,134580670,143689382,33342234,66973887,60404899,112714134,87501069,46918128,88344427
2024-01-01,146116727,165551924,154983524,63219630,157819495,146747835,35333836,67696347,76146671,150171679,98049678,47381687,92746810
2023-12-01,136834796,171819445,161333126,51955240,113176367,159890706,39484611,79379590,61749667,117283683,103173076,54080595,106043302
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-07-01,101472731,152932528,69859420,22181161,52197645,105028930,17394606,47199675,48226703,78860256,60524497,34658759,75195934
2014-06-01,109493957,153154138,78635596,23857473,57120898,106686443,19388706,55084733,52201231,82504171,63834491,40130672,78866783
2014-05-01,93088796,124731831,74951601,22594361,56134537,96743423,18072299,47752329,40417764,71626847,55169148,30099016,67334433


In [14]:
df_total = df_total_1+df_total_2

NameError: name 'df_total_1' is not defined

In [15]:
df_total = df_total_2

In [16]:
df_datetime = pd.DataFrame({'stt': [i for i in range(df_total.shape[0])]},
                          index=df_total.sort_index(ascending=True).index)
df_datetime['month'] = df_datetime.index.month
df_datetime['year'] = df_datetime.index.year
df_datetime

,stt,month,year
PHIEN,,,
2014-03-01,0,3,2014
2014-04-01,1,4,2014
2014-05-01,2,5,2014
2014-06-01,3,6,2014
2014-07-01,4,7,2014
...,...,...,...
2023-12-01,117,12,2023
2024-01-01,118,1,2024
2024-02-01,119,2,2024


In [18]:
! hdfs dfs -ls /path/to/company_data/user/<user>/cpc/data/parameter/

Found 7 items
-rwxrwx---+  3 <user> fdp     165099 2023-10-02 10:01 /path/to/company_data/user/<user>/cpc/data/parameter/QNGAI.parquet
-rwxrwx---+  3 <user> fdp    4871540 2023-10-02 17:30 /path/to/company_data/user/<user>/cpc/data/parameter/all_province_2017_2021.parquet
-rwxrwx---+  3 <user> fdp   16623237 2023-10-03 00:41 /path/to/company_data/user/<user>/cpc/data/parameter/all_province_2017_2021_v2.parquet
-rwxrwx---+  3 <user> fdp     679908 2023-10-06 18:01 /path/to/company_data/user/<user>/cpc/data/parameter/df_parameter_u2022.parquet
-rwxrwx---+  3 <user> fdp       5210 2023-10-05 10:37 /path/to/company_data/user/<user>/cpc/data/parameter/final_params_13province_u2022.parquet
-rwxrwx---+  3 <user> fdp       5225 2023-10-06 15:18 /path/to/company_data/user/<user>/cpc/data/parameter/final_params_13province_u2022_v2.parquet
-rwxrwx---+  3 <user> fdp       5829 2024-02-22 11:24 /path/to/company_data/user/<user>/cpc/data/parameter/final_params_13province_u2023.parquet


In [6]:
dfp = pd.read_parquet('/path/to/company_data/user/<user>/cpc/data/parameter/final_params_13province_u2023.parquet',filesystem=hdfs)
dfp

,province,n_max,lag,type_decomp,method_season,method_resid,method_trend,error_abs
0,BDINH,10,6,STL,statistics,None,average,0.081362
1,DANANG,10,6,STL,statistics,arima,average,0.066686
2,DLAC,10,6,STL,statistics,arima,average,0.198305
3,DNONG,5,6,STL,statistics,arima,average,0.246375
4,GLAI,5,8,STL,statistics,None,average,0.144923
5,KHOA,5,6,STL,statistics,arima,average,0.089738
6,KTUM,2,10,STL,statistics,None,average,0.128623
7,PYEN,5,6,STL,statistics,None,average,0.144254
8,QBINH,2,8,STL,statistics,None,average,0.141760
9,QNAM,10,6,STL,statistics,arima,average,0.066424


In [9]:
lst_prediction = pd.date_range(start='2022-01',end='2024-06',freq='MS').strftime('%Y-%m')
lst_prediction

Index(['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06',
       '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12',
       '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06',
       '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12',
       '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06'],
      dtype='object')

In [17]:
from tqdm import trange
# lst_pr = df_total.columns[:-2]
lst_pr = df_total.columns
df_result = pd.DataFrame()
years = [2022,2023,2024]

for n_province in trange(13):
    df = df_total.iloc[:, [n_province]]
    name_province = lst_pr[n_province]
    df = df.sort_index()
    dp = dfp[dfp.province == name_province]
    lag = 6
    n_max = dp.n_max.values[0]
    type_decomp = dp.type_decomp.values[0]
    method_season =dp.method_season.values[0]
    method_trend =dp.method_trend.values[0]
    method_resid =dp.method_resid.values[0]
    # n_max = 5
    # type_decomp = 'STL'
    # method_season='statistics'
    # method_trend='average'
    # method_resid='None'
    p = 1
    d = 0
    q = 1
    # print(name_province)
    # for year in years:
    #     for month in range(6, 13):
    for prediction_time in lst_prediction:
            year,month = prediction_time.split('-')
            year,month = int(year),int(month)

            # print(month)
            y_trend, y_seasonal, y_residual = predict_master(
                data=df,
                d_date=df_datetime,
                month=month, year=year,
                lag=lag, p=p, d=d, q=q,
                n_max=n_max,
                type_decomp=type_decomp,
                methods_season=method_season, method_resid=method_resid,
                method_trend=method_trend,order1=1,
                order2=1)

            y_pr = y_trend + y_seasonal + y_residual
            try:
                error, y_tr = compute_error(df, month, year, y_pr)
                df_result = pd.concat([df_result, pd.DataFrame(
                    {'n_max': n_max,
                     'province': name_province,
                     'year': year,
                     'month': month,
                     'y_pr': y_pr,
                     'y_tr': y_tr,
                     'error': error,
                     'error_abs': np.abs(error),
                     'y_trend': y_trend,
                     'y_seasonal': y_seasonal,
                     'y_residual': y_residual,
                     'lag': lag,
                     'p': p, 'd': d, 'q': q,
                     'type_decomp': type_decomp,
                     'method_season': method_season,
                     'method_resid': method_resid,
                     'method_trend': method_trend,
                     })])
            except:
                df_result = pd.concat([df_result, pd.DataFrame(
                    {'n_max': n_max,
                     'province': name_province,
                     'year': year,
                     'month': month,
                     'y_pr': y_pr,
                     'y_tr': [None],
                     'error': [None],
                     'error_abs': [None],
                     'y_trend': y_trend,
                     'y_seasonal': y_seasonal,
                     'y_residual': y_residual,
                     'lag': lag,
                     'p': p, 'd': d, 'q': q,
                     'type_decomp': type_decomp,
                     'method_season': method_season,
                     'method_resid': method_resid,
                     'method_trend': method_trend,
                     })])
                break


100%|██████████| 13/13 [00:20<00:00,  1.54s/it]


In [18]:
df_result['date'] = df_result.apply(lambda x:pd.to_datetime(f"{x.year}-{x.month}-01"),axis=1)
df_result

,n_max,province,year,month,y_pr,y_tr,error,error_abs,y_trend,y_seasonal,y_residual,lag,p,d,q,type_decomp,method_season,method_resid,method_trend,date
0,10,BDINH,2022,1,1.174573e+08,116642975,0.006981,0.006981,1.415852e+08,-2.412798e+07,0.0,6,1,0,1,STL,statistics,None,average,2022-01-01
0,10,BDINH,2022,2,1.154486e+08,120774882,-0.044101,0.044101,1.431311e+08,-2.768254e+07,0.0,6,1,0,1,STL,statistics,None,average,2022-02-01
0,10,BDINH,2022,3,1.206446e+08,117767281,0.024433,0.024433,1.459137e+08,-2.526908e+07,0.0,6,1,0,1,STL,statistics,None,average,2022-03-01
0,10,BDINH,2022,4,1.429444e+08,135568435,0.054408,0.054408,1.448183e+08,-1.873895e+06,0.0,6,1,0,1,STL,statistics,None,average,2022-04-01
0,10,BDINH,2022,5,1.404224e+08,143790595,-0.023424,0.023424,1.391127e+08,1.309687e+06,0.0,6,1,0,1,STL,statistics,None,average,2022-05-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,10,TTHUE,2024,1,1.056182e+08,92746810,0.138779,0.138779,1.192067e+08,-1.358858e+07,0.0,6,1,0,1,STL,statistics,None,average,2024-01-01
0,10,TTHUE,2024,2,1.042389e+08,88344427,0.179915,0.179915,1.190020e+08,-1.476307e+07,0.0,6,1,0,1,STL,statistics,None,average,2024-02-01
0,10,TTHUE,2024,3,9.872110e+07,103890813,-0.049761,0.049761,1.168254e+08,-1.810435e+07,0.0,6,1,0,1,STL,statistics,None,average,2024-03-01
0,10,TTHUE,2024,4,1.131949e+08,127099508,-0.109399,0.109399,1.203212e+08,-7.126252e+06,0.0,6,1,0,1,STL,statistics,None,average,2024-04-01


In [19]:
df_result.pivot_table(index='province',columns='date',values='error')

date,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2023-07-01,2023-08-01,2023-09-01,2023-10-01,2023-11-01,2023-12-01,2024-01-01,2024-02-01,2024-03-01,2024-04-01
province,,,,,,,,,,,,,,,,,,,,,
BDINH,0.006981,-0.044101,0.024433,0.054408,-0.023424,-0.029458,-0.069068,-0.027753,-0.005148,0.007390,...,-0.060347,-0.081501,-0.063380,-0.038347,-0.103458,0.123195,0.013926,0.039479,-0.011486,0.025446
DANANG,-0.059474,-0.108989,-0.028476,-0.058338,-0.043014,-0.075251,-0.094831,-0.081718,-0.102042,-0.008500,...,-0.010209,-0.061382,-0.064872,-0.124118,-0.057914,0.233091,0.222667,0.284832,-0.033621,-0.009745
DLAC,-0.004093,-0.067286,0.131343,0.182102,0.039595,-0.045767,-0.070810,-0.071275,-0.101659,-0.031070,...,-0.321757,0.132786,0.080926,0.047451,0.067100,-0.002554,0.034125,-0.030443,-0.087598,-0.033512
DNONG,0.484473,-0.038387,-0.179177,-0.275885,-0.147132,0.109912,0.046515,0.020986,-0.068868,-0.086137,...,0.198691,0.124978,0.072410,-0.007209,-0.114834,1.108063,-0.147154,-0.022744,-0.105922,0.000375
GLAI,0.061553,-0.061524,-0.052012,0.081547,0.061904,0.846505,0.148769,0.023295,-0.126000,-0.115221,...,0.175076,0.070998,-0.011816,-0.020179,-0.160338,-0.147493,-0.294916,0.110753,-0.028791,-0.011510
KHOA,-0.068087,-0.111279,-0.023649,-0.032891,-0.002798,-0.021077,-0.066472,0.008622,-0.012055,0.024149,...,-0.016150,-0.038963,-0.026362,-0.140991,-0.025027,0.067785,0.144757,0.165147,-0.043247,-0.017960
KTUM,-0.012480,-0.129880,-0.003268,0.001353,-0.681668,0.577034,0.165166,-0.050620,-0.145930,-0.052612,...,0.093198,0.041077,0.027683,-0.051326,-0.120452,-0.154036,-0.027768,0.150620,-0.124074,0.001514
PYEN,-0.062352,-0.191015,-0.040563,-0.045935,0.123296,0.019508,-0.070438,-0.048895,-0.074834,0.413918,...,-0.052875,-0.067886,-0.028375,0.026604,0.202376,-0.045793,0.082513,0.128782,-0.043920,-0.013323
QBINH,-0.033676,-0.050499,0.039288,0.004025,0.002550,0.136270,-0.056192,-0.111941,-0.090189,-0.017306,...,-0.067224,0.019522,0.148691,0.013849,-0.110494,0.205074,-0.065062,0.234477,-0.026673,-0.151707


In [54]:
df_result[['province','year','month','date','y_pr','y_tr','error',]]

,province,year,month,date,y_pr,y_tr,error
0,BDINH,2022,1,2022-01-01,1.768483e+08,173882671,0.017055
0,BDINH,2022,2,2022-02-01,1.652732e+08,168962063,-0.021833
0,BDINH,2022,3,2022-03-01,1.802569e+08,184682976,-0.023966
0,BDINH,2022,4,2022-04-01,2.026058e+08,199514144,0.015496
0,BDINH,2022,5,2022-05-01,1.987774e+08,205777410,-0.034017
...,...,...,...,...,...,...,...
0,TTHUE,2024,1,2024-01-01,1.544538e+08,143823830,0.073910
0,TTHUE,2024,2,2024-02-01,1.487777e+08,127712344,0.164943
0,TTHUE,2024,3,2024-03-01,1.589985e+08,163354735,-0.026667
0,TTHUE,2024,4,2024-04-01,1.736174e+08,186665197,-0.069899


In [56]:
df_result[['province','year','month','date','y_pr','y_tr','error',]].to_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr12_1t_sum_13province_20240501_v2023.parquet',filesystem=hdfs)

In [57]:
pd.read_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr12_1t_sum_13province_20240501_v2023.parquet',filesystem=hdfs).filter(['province','year','month','date','y_pr','y_tr','error'])

,province,year,month,date,y_pr,y_tr,error
0,BDINH,2022,1,2022-01-01,1.768483e+08,173882671.0,0.017055
0,BDINH,2022,2,2022-02-01,1.652732e+08,168962063.0,-0.021833
0,BDINH,2022,3,2022-03-01,1.802569e+08,184682976.0,-0.023966
0,BDINH,2022,4,2022-04-01,2.026058e+08,199514144.0,0.015496
0,BDINH,2022,5,2022-05-01,1.987774e+08,205777410.0,-0.034017
...,...,...,...,...,...,...,...
0,TTHUE,2024,1,2024-01-01,1.544538e+08,143823830.0,0.073910
0,TTHUE,2024,2,2024-02-01,1.487777e+08,127712344.0,0.164943
0,TTHUE,2024,3,2024-03-01,1.589985e+08,163354735.0,-0.026667
0,TTHUE,2024,4,2024-04-01,1.736174e+08,186665197.0,-0.069899


In [87]:
df_result[df_result.date=='2023-09-01'].groupby(['province']).agg({'error_abs':'mean'})

,error_abs
province,
BDINH,0.080369
DANANG,0.044973
DLAC,0.062864
DNONG,0.093576
GLAI,0.036680
KHOA,0.018424
KTUM,0.025406
PYEN,0.033219
QBINH,0.050106


In [54]:
pd.read_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr2_1t_sum_13province_20240305_v2022.parquet',filesystem=hdfs)

,province,year,month,date,y_pr,y_tr,error
0,BDINH,2022,1,2022-01-01,1.148081e+08,115747365.0,-0.008115
0,BDINH,2022,2,2022-02-01,1.102128e+08,120023398.0,-0.081739
0,BDINH,2022,3,2022-03-01,1.192655e+08,116855829.0,0.020621
0,BDINH,2022,4,2022-04-01,1.444004e+08,134581192.0,0.072961
0,BDINH,2022,5,2022-05-01,1.394165e+08,142848913.0,-0.024028
...,...,...,...,...,...,...,...
0,TTHUE,2023,11,2023-11-01,1.146642e+08,117468173.0,-0.023870
0,TTHUE,2023,12,2023-12-01,1.129682e+08,117866864.0,-0.041561
0,TTHUE,2024,1,2024-01-01,1.067941e+08,97280619.0,0.097794
0,TTHUE,2024,2,2024-02-01,1.134844e+08,93030330.0,0.219865


In [58]:
pd.read_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr1_12t_sum_13province_20240305_v2022.parquet',filesystem=hdfs)

,province,year,month,y_pr1,y_pr2,y_pr3,y_pr4,y_pr5,y_pr6,y_pr7,y_pr8,y_pr9,y_pr10,y_pr11,y_pr12
0,BDINH,2024,1,5.541403e+07,4.627546e+07,6.783818e+07,6.483950e+07,6.198355e+07,6.065635e+07,6.237334e+07,6.613373e+07,6.598100e+07,6.751752e+07,6.758981e+07,7.087016e+07
0,DANANG,2024,1,7.653139e+07,6.741431e+07,8.077517e+07,8.485725e+07,8.537875e+07,9.328270e+07,9.205592e+07,9.316524e+07,8.726853e+07,8.575670e+07,8.281280e+07,7.651101e+07
0,DLAC,2024,1,3.186770e+07,3.148685e+07,3.993107e+07,2.912567e+07,2.624981e+07,2.591652e+07,2.560235e+07,2.589041e+07,3.140704e+07,2.665417e+07,2.936294e+07,3.145209e+07
0,DNONG,2024,1,1.529053e+07,1.234068e+07,1.641660e+07,1.292347e+07,1.236989e+07,1.212497e+07,1.362244e+07,1.404884e+07,1.672816e+07,1.374162e+07,1.553757e+07,1.733896e+07
0,GLAI,2024,1,1.227361e+07,1.271809e+07,1.367206e+07,1.148751e+07,8.134227e+06,7.170300e+06,6.094937e+06,7.189163e+06,1.093675e+07,1.145178e+07,1.142133e+07,1.115474e+07
0,KHOA,2024,1,6.254059e+07,5.974517e+07,6.152946e+07,6.767783e+07,6.902471e+07,7.660965e+07,7.146519e+07,7.102494e+07,6.660747e+07,6.650734e+07,6.534397e+07,6.193620e+07
0,KTUM,2024,1,1.121191e+07,1.043294e+07,1.220639e+07,7.018979e+06,3.766658e+06,3.953035e+06,4.783398e+06,1.014344e+07,1.102319e+07,1.230240e+07,1.136914e+07,1.455904e+07
0,PYEN,2024,1,9.878109e+06,1.005865e+07,1.239089e+07,1.130879e+07,1.011755e+07,1.025364e+07,1.050487e+07,1.098229e+07,1.254385e+07,1.345923e+07,1.341455e+07,1.267226e+07
0,QBINH,2024,1,9.030051e+06,7.791450e+06,1.418239e+07,1.590634e+07,1.377516e+07,1.411000e+07,1.516664e+07,1.512979e+07,1.620351e+07,1.403045e+07,1.574777e+07,1.627812e+07
0,QNAM,2024,1,5.872885e+07,5.396354e+07,6.167874e+07,6.492429e+07,6.367007e+07,6.628129e+07,6.837077e+07,6.823034e+07,6.631747e+07,6.794450e+07,6.702686e+07,6.635165e+07
